In [3]:
import os, sys
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
import os, sys, glob, pickle
from functools import partial

import jax
import jax.numpy as jnp
from jax.random import split
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from einops import rearrange, reduce, repeat

import substrates
import foundation_models
from rollout import rollout_simulation
import asal_metrics
import util

In [ ]:
! python main_opt.py --seed=0 --save_dir="./data/supervised_0" --substrate="lenia" --time_sampling=1 --prompts="a glider" --coef_prompt=1. --coef_softmax=0. --coef_oe=0. --bs=1 --pop_size=16 --n_iters=1000 --sigma=0.1

In [ ]:
save_dir = "./data/supervised_0"
data = util.load_pkl(save_dir, "data") # load optimization data
params, best_loss = util.load_pkl(save_dir, "best") # load the best parameters found

# fm = foundation_models.create_foundation_model('clip') # we don't need the foundation model for just the rollout currently
substrate = substrates.create_substrate('lenia') # create the substrate
substrate = substrates.FlattenSubstrateParameters(substrate) # useful wrapper to flatten the substrate parameters

rollout_fn = partial(rollout_simulation, s0=None, substrate=substrate, fm=None, rollout_steps=substrate.rollout_steps, time_sampling=8, img_size=224, return_state=False)
rollout_fn = jax.jit(rollout_fn)

rng = jax.random.PRNGKey(0)
rollout_data = rollout_fn(rng, params) # rollout the simulation using this rng seed and simulation parameters